## กีฬาสี

### โจทย์ บริษัทแห่งหนึ่งจัดงานกีฬาสีเชื่อมความสัมพันธ์โดยใช้วิธี Genetic
- สีแบ่งออกเป็น $สีแดง$ | $สีน้ำเงิน$ 
- ต้องมีความเท่าเทียม เช่น เพศ อายุ น้ำหนัก ส่วนสูง
- ต้องคละ department เท่า ๆ กันในแต่ละสี
### ผลลัพที่ต้องการ
- visturization ของแต่ละคนในแต่ละสี ต้องระบุว่าสีนั้น ๆ มีพนักงานคนั้นมีใครบ้าง

# ขั้นตอนการทำ

#### import ไรเบอรี่ที่ใช้

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

#### ให้ employee_id กับ พนักงาน

In [21]:
# โหลดชุดข้อมูล HR.csv
df = pd.read_csv('HR.csv')

# ตัดช่องว่างหัวท้ายข้อความของคอลัมน์สตริง
df['gender'] = df['gender'].astype(str).str.strip()
df['department'] = df['department'].astype(str).str.strip()

# สร้าง employee_id ให้พนักงานแต่ละคน (เช่น EMP_001, EMP_002, ...)
df.insert(0, 'employee_id', [f'EMP_{i+1:03d}' for i in range(len(df))])

# แสดงจำนวนและตัวอย่างข้อมูล
print(f'จำนวนพนักงานทั้งหมด: {len(df)} คน')
df.head(10)

จำนวนพนักงานทั้งหมด: 358 คน


,employee_id,gender,department,weight,height,age
0,EMP_001,Male,R,89,172,32
1,EMP_002,Male,R,89,180,52
2,EMP_003,Male,R,52,158,27
3,EMP_004,Male,R,50,157,39
4,EMP_005,Male,R,68,174,25
5,EMP_006,Male,R,62,164,29
6,EMP_007,Male,R,74,168,53
7,EMP_008,Male,R,64,164,47
8,EMP_009,Male,R,58,166,60
9,EMP_010,Female,R,81,178,36


#### กำหนด DNA

- จากทั้งหมด 358 คน แบ่งออกไก้เป็น สีแดง 179(0) | สีน้ำเงิน 179(1) 
- ให้ยืนมีแค่าเป็น {Ci1, Ci2, ..., Cin} Ci คือ กลุ่มทีมของพนักงานคนที่ i มีค่าเป็น 0 และ 1

In [22]:
'''
# สร้างโครโมโซมเริ่มต้นแบบสุ่ม size ชุด แต่ละชุดยาว len(df) การันตีอัตราส่วน 0:1 = 179:179
def generate_pop_simetry(size, n_employees = len(df)):

    #แบ่งจำนวนของแต่ละทีมที่ควรมี
    NUM_RED = n_employees // 2        
    NUM_BLUE = n_employees - NUM_RED   
    # สร้าง [0 1] ตามจำนวนที่กำหนดแบบเรียงกัน
    base_dna = np.array([0] * NUM_RED + [1] * NUM_BLUE) 
    # คืนค่าเป็นการสลับแบบสุ่มโดยผลรวมยังเท่ากัน
    return np.array([np.random.permutation(base_dna) for _ in range(size)])
'''

# สร้างโครโมโซมเริ่มต้นแบบสุ่ม size ชุด แต่ละชุดยาว len(df) แบบสุ่มทั้งหมด (0 หรือ 1 อิสระ)
def generate_pop_asimetry(size, n_employees = len(df)):
    return np.random.randint(0, 2, size=(size, n_employees))

# สร้างประชากรเริ่มต้น 100 คน
pop = generate_pop_asimetry(100)

print(f"ขนาดประชากร: {pop.shape}")  # (100, 358)
print("ตัวอย่าง DNA ตัวแรก: \n", pop[0])


ขนาดประชากร: (100, 358)
ตัวอย่าง DNA ตัวแรก: 
 [1 1 0 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 1
 0 0 1 1 0 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 1 1 1 0 1 0 0 0 1 1 0 1 0 1 1
 0 1 1 0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 1 1 0 1 1 0 1 1 0 1 0 0 0 0 0 1 0 1
 0 0 1 1 1 1 0 0 0 0 1 0 0 0 1 0 1 0 1 0 0 1 1 0 0 1 0 0 1 0 1 0 0 0 1 0 1
 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 1 1 1 1 0 0 0 1 0 1 1 1 0 0 0 0 0 1 1 1 0
 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 1 0 1 1 1 0 1 0 0 0 1 0 0 1
 0 1 0 1 1 1 1 1 1 0 1 1 0 1 1 0 0 0 0 1 0 0 0 0 1 1 0 0 1 0 1 1 0 0 1 0 1
 1 1 0 1 1 1 0 0 0 1 0 0 0 1 0 1 1 1 0 1 1 1 0 1 1 0 1 1 0 1 0 0 0 1 0 1 0
 1 0 0 1 1 0 0 0 1 0 0 1 1 1 1 0 0 0 1 1 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 1 0
 1 0 1 0 0 1 1 0 1 1 0 0 1 1 1 1 0 1 1 1 0 0 0 1 1]


#### วัดผล
- กลุ่มจำนวนต่อเนื่อง
    - จำนวน : N(0) - N(1) สำคัญมาก *10 
    - อายุ : Mean(อายุ 0) - Mean(อายุ 1) 
    - น้ำหนัก : Mean(น้ำหนัก 0) - Mean(น้ำหนัก 1) 
    - ส่วนสูง : Mean(ส่วนสูง 0) - Mean(ส่วนสูง 1) 
- กลุ่มการกระจายตัว
    - เพศ : N(ผู้ชาย 0) - N(ผู้ชาย 1) ที่เหลือเป้นผู้หญิง
    - แผนก : หาผลรวมของ N(Di0) - (Di1) โดย Di คือแผนกนั้น ๆ

In [23]:
# แปลงข้อมูลเป็น NumPy Array ล่วงหน้า เพื่อให้คำนวณได้รวดเร็ว
weights = df['weight'].values
heights = df['height'].values
ages = df['age'].values
is_male = (df['gender'] == 'Male').values.astype(int)
dept_dummies = pd.get_dummies(df['department']).values

# input: โครโมโซม 1 ตัว 
# process: คำนวณความไม่เท่าเทียม (จำนวนคน, อายุ, น้ำหนัก, ส่วนสูง, เพศ, แผนก)
# output: ความไม่เท่าเทียม total_error (ยิ่งน้อยยิ่งดี)
def evaluate_single(dna):
    red = (dna == 0)
    blue = (dna == 1)
    
    n_red = red.sum()
    n_blue = blue.sum()
    if n_red == 0 or n_blue == 0:
        return 1e9
    
    # ผลต่างจำนวนคน
    diff_count = abs(n_red - n_blue)*10 #เงื่อนไงสำคัญ
    
    # ผลต่างค่าเฉลี่ย อายุ, น้ำหนัก, ส่วนสูง
    diff_age = abs(ages[red].mean() - ages[blue].mean())
    diff_weight = abs(weights[red].mean() - weights[blue].mean())
    diff_height = abs(heights[red].mean() - heights[blue].mean())
    
    # ผลต่างจำนวนเพศชายระหว่างสองสี
    diff_gender = abs(is_male[red].sum() - is_male[blue].sum())
    
    # ผลรวมของผลต่างการกระจายตัวของแต่ละแผนก
    diff_dept = np.sum(np.abs(dept_dummies[red].sum(axis=0) - dept_dummies[blue].sum(axis=0)))
    
    # รวมคะแนนความต่าง
    total_error = diff_count + diff_age + diff_weight + diff_height + diff_gender + diff_dept
    return total_error

# ฟังก์ชันวัดผลของประชากรทั้งชุด
def fitness(pop):
    scores = []
    for dna in pop:
        score = evaluate_single(dna)
        scores.append(score)
    return np.array(scores)

# ทดสอบวัดผลประชากรเริ่มต้น
f = fitness(pop)
best_score = f.min()
print('Fitness ที่ดีที่สุดในรุ่นเริ่มต้น :', best_score)


Fitness ที่ดีที่สุดในรุ่นเริ่มต้น : 66.52197253433208


#### สืบพันธู์ประชากรทั้งแบบ$อาศัยเพศ$และแบบ$ไม่อาศัยเพศ$

In [24]:
# input: พ่อ (p1) และ แม่ (p2) | process: สุ่มจุดตัดแล้วนำยีนพ่อและแม่มาต่อกันตรงๆ | output: โครโมโซมลูก
def sexual(p1, p2):
    c = np.random.randint(1, len(p1))  # สุ่มตำแหน่งจุดตัด 
    child = np.concatenate((p1[:c], p2[c:]))  # ครึ่งแรกจากพ่อ + ครึ่งหลังจากแม่
    return child

# input: โครโมโซม 1 ตัว (p) | process: สุ่มสลับสีพนักงาน 1 ตำแหน่ง (0 <-> 1) | output: โครโมโซมที่กลายพันธุ์แล้ว
def asexual(p):
    child = p.copy()  # ทำสำเนาเพื่อไม่ให้กระทบตัวเดิม
    i = np.random.randint(0, len(child))  # สุ่มเลือกพนักงาน 1 คน
    child[i] = 1 - child[i]               # สลับสี: ถ้า 0 เปลี่ยนเป็น 1, ถ้า 1 เปลี่ยนเป็น 0
    return child

# ทดสอบฟังก์ชันทั้งสองแบบ
test_child = sexual(pop[0], pop[1])
test_mutated = asexual(test_child)
print(f"ผลลัพธ์ Crossover: แดง {(test_child == 0).sum()} คน, น้ำเงิน {(test_child == 1).sum()} คน (ความยาว {len(test_child)})")
print(f"ผลลัพธ์ Mutation:  แดง {(test_mutated == 0).sum()} คน, น้ำเงิน {(test_mutated == 1).sum()} คน (ความยาว {len(test_mutated)})")


ผลลัพธ์ Crossover: แดง 198 คน, น้ำเงิน 160 คน (ความยาว 358)
ผลลัพธ์ Mutation:  แดง 197 คน, น้ำเงิน 161 คน (ความยาว 358)


#### สร้าง Visturization

In [ ]:
def visturization():
    # เขียนตรงนี้งับ :)
    pass

#### Loop ปล่อยให้ธรรมชาติคัดสรร

In [ ]:
# ฟังก์ชันตรวจสอบเงื่อนไขการหยุด พร้อมพิมพ์สรุปผลลัพธ์การแบ่งสี
def is_stop(gen, max_gen, pop, best_score):
    if gen >= max_gen:
        best = pop[0]
        red = (best == 0)
        blue = (best == 1)
        
        diff_age = abs(ages[red].mean() - ages[blue].mean())
        diff_weight = abs(weights[red].mean() - weights[blue].mean())
        diff_height = abs(heights[red].mean() - heights[blue].mean())
        diff_gender = abs(is_male[red].sum() - is_male[blue].sum())
        diff_dept = np.sum(np.abs(dept_dummies[red].sum(axis=0) - dept_dummies[blue].sum(axis=0)))
        
        print(f"\n🎉 สิ้นสุดการทำงานที่รุ่น {gen} | fitness ที่ดีที่สุด = {best_score:.4f}")
        print(f"พนักงานจำนวน {len(df)} คน แบ่งออกเป็น สีแดง {red.sum()} คน | สีน้ำเงิน {blue.sum()} คน")
        print(f"ผลต่างค่าเฉลี่ยอายุ: {diff_age:.2f} ปี (แดง: {ages[red].mean():.1f} | น้ำเงิน: {ages[blue].mean():.1f})")
        print(f"ผลต่างค่าน้ำหนัก: {diff_weight:.2f} กก. (แดง: {weights[red].mean():.1f} | น้ำเงิน: {weights[blue].mean():.1f})")
        print(f"ผลต่างค่าส่วนสูง: {diff_height:.2f} ซม. (แดง: {heights[red].mean():.1f} | น้ำเงิน: {heights[blue].mean():.1f})")
        print(f"ผลต่างเพศชาย: {diff_gender} คน (แดง: {is_male[red].sum()} | น้ำเงิน: {is_male[blue].sum()})")
        print(f"ผลต่างการกระจายแผนก: {diff_dept} คน (จาก {df['department'].nunique()} แผนก)")
        return True
    return False

# ฟังก์ชันการคัดสรรทางธรรมชาติ (Genetic Algorithm Loop)
def natural_selection(pop, max_gen=500, k=50, mutation_rate=0.8):
    gen = 0
    while True:
        # วัดผล 
        f = fitness(pop)
        
        # คัดเลือก: จัดลำดับจากคะแนนดีที่สุดไปแย่ที่สุด
        i = np.argsort(f)
        pop = pop[i]
        best_score = f[i[0]]
        
        # แสดงผลทุก 50 รุ่น
        if gen % 50 == 0:
            visturization() # <-------------------------------------------- ใส่พารามิเตอร์ด้วย
            print(f"รุ่นที่ {gen:4d} | fitness = {best_score:.4f}")
            
        # ตรวจสอบเงื่อนไขการหยุด
        if is_stop(gen, max_gen, pop, best_score):
            break
            
        gen += 1
        
        # สืบพันธุ์: สุ่มพ่อแม่จากกลุ่ม Top k มาสร้างลูกแทนที่ครึ่งหลัง
        for idx in range(k, len(pop)):
            p1 = pop[np.random.randint(0, k)]
            p2 = pop[np.random.randint(0, k)]
            child = sexual(p1, p2)
            
            # มีโอกาสกลายพันธุ์ 
            if np.random.rand() < mutation_rate:
                child = asexual(child)
                
            pop[idx] = child

    best_dna = pop[0]
    return best_dna

# รันฟังก์ชันเพื่อหาคำตอบที่ดีที่สุด
pop = generate_pop_asimetry(100)
best_dna = natural_selection(pop)

รุ่นที่    0 | fitness = 63.3017
รุ่นที่   50 | fitness = 25.6592
รุ่นที่  100 | fitness = 11.9050
รุ่นที่  150 | fitness = 6.3408
รุ่นที่  200 | fitness = 6.3408
รุ่นที่  250 | fitness = 6.3408
รุ่นที่  300 | fitness = 6.3408
รุ่นที่  350 | fitness = 6.3408
รุ่นที่  400 | fitness = 6.3408
รุ่นที่  450 | fitness = 6.3408
รุ่นที่  500 | fitness = 6.3408

🎉 สิ้นสุดการทำงานที่รุ่น 500 | fitness ที่ดีที่สุด = 6.3408
พนักงานจำนวน 358 คน แบ่งออกเป็น สีแดง 179 คน | สีน้ำเงิน 179 คน
ผลต่างค่าเฉลี่ยอายุ: 0.27 ปี (แดง: 41.4 | น้ำเงิน: 41.1)
ผลต่างค่าน้ำหนัก: 0.06 กก. (แดง: 72.1 | น้ำเงิน: 72.1)
ผลต่างค่าส่วนสูง: 0.01 ซม. (แดง: 167.9 | น้ำเงิน: 167.9)
ผลต่างเพศชาย: 0 คน (แดง: 61 | น้ำเงิน: 61)
ผลต่างการกระจายแผนก: 6 คน (จาก 19 แผนก)
